# Attention Pattern & RoPE Heatmap Visualization

This notebook visualises what a trained GPT model "sees" inside its attention heads.

### What you'll see

| Visualization | What it shows |
|---|---|
| Attention heatmap per head | Which tokens each token attends to |
| Head aggregation | Average attention pattern across all heads |
| RoPE rotation matrix | How RoPE modifies the Q·K dot product by position |
| RoPE frequency bands | The multi-resolution structure of rotary encoding |

### Why this matters

Attention patterns reveal **what the model has learned** about language structure:
- Some heads attend to the **previous token** (syntactic locality)
- Some heads attend to **distant context** (long-range dependencies)
- Some heads learn **specialized roles** (punctuation, verbs, etc.)
- RoPE ensures position information is **smoothly encoded** across frequency bands

In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parent) if Path.cwd().name == "apps" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.append(project_root)

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from core.transformer import (
    GPT,
    BPETokenizer,
    RotaryEmbedding,
    apply_rotary_emb,
)

print("All imports loaded.")
print(
    f"PyTorch {torch.__version__}, device: {'cuda' if torch.cuda.is_available() else 'cpu'}"
)

---
## 0. Model Setup

Load your trained GPT or instantiate a small one for structure analysis.
Even an untrained model produces interpretable attention patterns because
the causal mask and RoPE impose a strong structural prior.

In [ ]:
# ── Model configuration ─────────────────────────────────────────────────
# Load the trained model config, tokenizer, and weights from models/

MODEL_DIR = Path(project_root) / "models"

config = torch.load(
    MODEL_DIR / "training_config.pt", map_location="cpu", weights_only=True
)
print("=== Training Config ===")
for k, v in config.items():
    print(f"  {k}: {v}")

# ── Load BPE tokenizer ─────────────────────────────────────────────────
tokenizer_data = torch.load(
    MODEL_DIR / "bpe_tokenizer.pt", map_location="cpu", weights_only=True
)
tokenizer = BPETokenizer(
    vocab_size=len(tokenizer_data["vocab"]),
    special_tokens=tokenizer_data["special_tokens"],
    regex_pattern=tokenizer_data["regex_pattern"],
)
tokenizer.vocab = tokenizer_data["vocab"]
tokenizer.id_to_token = tokenizer_data["id_to_token"]
tokenizer.merges = tokenizer_data["merges"]
tokenizer.merge_ranks = tokenizer_data["merge_ranks"]
print(f"\nBPE tokenizer: vocab_size={tokenizer.vocab_size}")

# ── Create model with saved config and load trained weights ─────────────
# Phase 6: pass through GQA (n_kv_heads) and MoE params if present in config
model = GPT(
    vocab_size=config["vocab_size"],
    d_model=config["d_model"],
    n_layers=config["n_layers"],
    n_heads=config["n_heads"],
    n_kv_heads=config.get("n_kv_heads"),
    max_seq_len=config["block_size"],
    d_ff=config["d_ff"],
    dropout=0.0,  # disable dropout for analysis (inference mode)
    use_moe=config.get("use_moe", False),
    n_experts=config.get("n_experts", 8),
    moe_k=config.get("moe_k", 2),
)
model.load_state_dict(
    torch.load(
        MODEL_DIR / "gpt_tinyshakespeare.pt", map_location="cpu", weights_only=True
    )
)
model.eval()

print(f"\nModel: {model}")
print(f"Parameters: {model.num_parameters:,}")

# ── Store constants for later use ───────────────────────────────────────
VOCAB_SIZE = config["vocab_size"]
D_MODEL = config["d_model"]
N_LAYERS = config["n_layers"]
N_HEADS = config["n_heads"]
BLOCK_SIZE = config["block_size"]
D_FF = config["d_ff"]
D_K = D_MODEL // N_HEADS  # per-head dimension used by RoPE

In [ ]:
# ── Sample prompt ──────────────────────────────────────────────────────

prompt = "ROMEO: But, soft! what light through yonder window breaks?"
ids = tokenizer.encode(prompt)
token_ids = torch.tensor([ids], dtype=torch.long)

# Show BPE token-level decomposition
tokens = [tokenizer.id_to_token[i] for i in ids]
print(f"Prompt: {prompt}")
print(f"BPE tokens ({len(tokens)}): {tokens}")
print(f"Token IDs shape: {token_ids.shape}")

---
## 1. Extracting Attention Weights

The standard `GPT.forward()` returns only logits — the attention weights
are computed inside `scaled_dot_product_attention()` and then discarded.

To capture them, we use **PyTorch forward hooks** — functions that are
called automatically when a module's forward pass runs:

```python
def hook(module, input, output):
    # input  = tuple of args to the module's forward
    # output = what the module returns
    ...
handle = module.register_forward_hook(hook)
```

We'll register a hook on each `GPTBlock.attn` (`MultiHeadAttention`)
to capture the attention probability matrix.

### Problem: The weights are inside `scaled_dot_product_attention()`

The weights `softmax(QK^T / sqrt(d_k))` are computed and consumed inside
a utility function, not returned. We have two options:
1. Modify `scaled_dot_product_attention` to return weights (invasive)
2. **Register a hook on `F.softmax` itself** (non-invasive, hackish)
3. **Replace the attention function temporarily** — create a patched version
   that stores the weights on the module as a side effect

We'll go with option 3 — patching the call inside MultiHeadAttention.forward.
Alternatively, a simpler approach: **run a single-head, single-layer forward
and manually compute attention** inside this notebook.

---
**Simplest approach for this notebook:** Call attention manually on each
block's Q, K, V to get the weight matrix.

In [ ]:
# ── Extract attention weights from all layers and heads ────────────────


def get_attention_weights(model, token_ids):
    """Run forward pass and collect attention weights from every layer.

    Uses monkey-patching to intercept the softmax inside
    scaled_dot_product_attention.

    Returns
    -------
    attn_weights : list[torch.Tensor]
        One tensor per layer: (n_heads, seq_len, seq_len).
        Each entry is the attention probability matrix A where
        A[h, i, j] = how much head h at position i attends to position j.
    """
    import math

    from core.transformer.transformer import scaled_dot_product_attention as original_fn

    captured_weights = []

    def patched_attention(q, k, v, mask=None):
        """Identical to original, but captures attention weights."""
        d_k = q.size(-1)
        scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        captured_weights.append(weights.detach().cpu())
        return weights @ v

    # ── Monkey-patch ─────────────────────────────────────────────────────
    import core.transformer.transformer as transformer_mod

    transformer_mod.scaled_dot_product_attention = patched_attention

    with torch.no_grad():
        # This forward pass calls scaled_dot_product_attention once per
        # layer inside MultiHeadAttention.forward. Each call triggers our
        # patched version, which appends the attention weights to the list.
        _ = model(token_ids)

    # ── Restore original ─────────────────────────────────────────────────
    transformer_mod.scaled_dot_product_attention = original_fn

    return captured_weights


# ── Run extraction ─────────────────────────────────────────────────────
attn_weights = get_attention_weights(model, token_ids)
print(f"Captured {len(attn_weights)} layers of attention weights")
print(f"Each shape: {list(attn_weights[0].shape)}")

---
## 2. Attention Heatmap Visualization

Each attention head produces a matrix $A \in [0, 1]^{T \times T}$ where:

- $A[i, j]$ = how much token $i$ attends to token $j$
- Rows sum to 1 (softmax normalisation)
- Upper triangle = 0 (causal mask: token $i$ can't see token $j > i$)
- Diagonal = self-attention

### What to look for

- **Sharp diagonal**: token mostly attends to itself (or nearby tokens)
- **Broad column**: distributed attention across many tokens
- **Vertical stripes**: a token is universally attended-to (e.g. a period)
- **Structured bands**: syntactic patterns (verbs attending to subjects)

In [ ]:
# ── Plot attention heatmaps ────────────────────────────────────────────
# Each layer takes HEADS_PER_ROW columns, so a layer with 8 heads
# occupies 2 rows × 4 columns.  This gives each head enough room.

HEADS_PER_ROW = 4  # ← change this to control subplot width

seq_len = token_ids.size(1)
n_heads = N_HEADS  # from model config loaded above

if len(attn_weights) == 0:
    raise RuntimeError(
        "No attention weights captured. Run get_attention_weights() first."
    )

print(f"Plotting {len(attn_weights)} layers × {n_heads} heads, seq_len={seq_len}")
print(f"Sample shape: {attn_weights[0].shape}")

rows_per_layer = (n_heads + HEADS_PER_ROW - 1) // HEADS_PER_ROW  # ceil division
total_rows = N_LAYERS * rows_per_layer

fig, axes = plt.subplots(
    total_rows,
    HEADS_PER_ROW,
    figsize=(
        2.5 * HEADS_PER_ROW,
        2.5 * total_rows,
    ),  # slightly larger for BPE token labels
)
# Make axes always 2D so we can index [row, col] safely
if total_rows == 1:
    axes = axes.reshape(1, -1)
elif HEADS_PER_ROW == 1:
    axes = axes.reshape(-1, 1)

for layer_idx in range(N_LAYERS):
    for head_idx in range(n_heads):
        # Map (layer, head) → (row, col) in the compact grid
        row = layer_idx * rows_per_layer + head_idx // HEADS_PER_ROW
        col = head_idx % HEADS_PER_ROW
        ax = axes[row, col]

        weights = attn_weights[layer_idx][0, head_idx]  # (T, T)

        ax.imshow(weights, cmap="Blues", aspect="auto", vmin=0, vmax=weights.max())

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f"L{layer_idx} H{head_idx}", fontsize=9)

        # BPE token labels for short sequences
        if seq_len <= 40:
            ax.set_xticks(range(seq_len))
            ax.set_yticks(range(seq_len))
            ax.set_xticklabels(tokens, fontsize=5, rotation=90)
            ax.set_yticklabels(tokens, fontsize=5)

# Hide any leftover empty subplots
n_total_subplots = total_rows * HEADS_PER_ROW
n_actual_subplots = N_LAYERS * n_heads
for idx in range(n_actual_subplots, n_total_subplots):
    row = idx // HEADS_PER_ROW
    col = idx % HEADS_PER_ROW
    axes[row, col].set_visible(False)

fig.suptitle("Attention Weights per Layer × Head", fontsize=13, y=1.02)
fig.subplots_adjust(left=0.06, right=0.96, wspace=0.35, hspace=0.5)
plt.show()

### Aggregated Attention Patterns

Average across all heads to see the **consensus attention pattern**.
This reveals whether certain token positions systematically attract
more attention across the whole model.

In [ ]:
# ── Average attention across all heads ──────────────────────────────────
# Shows the model's "receptive field" — how local vs global attention is.

seq_len = token_ids.size(1)

# Stack all attention weights, squeezing batch dim: (L, H, T, T)
all_weights = torch.cat(
    attn_weights, dim=0
)  # (L, H, T, T) — since each is (1, H, T, T)

# Average across ALL layers and heads
avg_attention = all_weights.mean(dim=(0, 1))  # (T, T)

# ── Heatmap ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
im = ax.imshow(avg_attention, cmap="viridis", aspect="auto")
ax.set_title("Average Attention Across All Heads")
ax.set_xlabel("Source Position (j)")
ax.set_ylabel("Target Position (i)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

if seq_len <= 40:
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.set_xticklabels(tokens, fontsize=5, rotation=90)
    ax.set_yticklabels(tokens, fontsize=5)

# ── Attention vs distance profile ──────────────────────────────────────
ax = axes[1]
weights_sum = torch.zeros(seq_len)
counts = torch.zeros(seq_len)
for i in range(seq_len):
    for j in range(i + 1):  # only look at allowed positions (j <= i, causal)
        d = i - j
        weights_sum[d] += avg_attention[i, j].item()
        counts[d] += 1
avg_by_distance = weights_sum / counts.clamp(min=1)

distances = torch.arange(seq_len)
ax.plot(distances, avg_by_distance, "o-", markersize=3, linewidth=1)
ax.set_xlabel("Distance (|i - j|)")
ax.set_ylabel("Average Attention Weight")
ax.set_title("Attention Decay with Distance")
ax.set_xlim(0, min(seq_len - 1, 50))  # zoom in on first 50 positions
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Attention at distance 0 (self):  {avg_by_distance[0]:.4f}")
print(
    f"Attention at distance 1:         {avg_by_distance[1] if seq_len > 1 else 'N/A':.4f}"
)
print(
    f"Attention at distance 10:        {avg_by_distance[10] if seq_len > 10 else 'N/A':.4f}"
)
if seq_len > 2:
    print(
        f"Tail average (dist > {seq_len // 2}):     {avg_by_distance[seq_len // 2 :].mean():.4f}"
    )

---
## 3. RoPE Heatmap Visualization

Rotary Position Encoding works by rotating Q and K vectors in 2D subspaces.

### Mathematical Recap

For position $m$ and dimension pair $i$, the rotation angle is:

$$\theta_i = 10000^{-2i / d}$$
$$\text{RoPE}(x_m)_i = x_m \cdot \cos(m\theta_i) + \text{rotate\_half}(x_m) \cdot \sin(m\theta_i)$$

### Key Property

After RoPE, the dot product between Q at position $m$ and K at position $n$ depends
**only on their relative position** $(m-n)$ for each frequency band:

$$\langle \text{RoPE}(q_m), \text{RoPE}(k_n) \rangle = f(q, k, m-n)$$

Let's visualise:
1. The rotation angles $\theta_i$ across dimensions (frequency bands)
2. The rotation matrix as a heatmap over positions
3. The implicit **relative position bias** — how RoPE makes $Q \cdot K$ decay with distance

In [ ]:
# ── Precompute RoPE frequencies ────────────────────────────────────────
D_K = D_MODEL // N_HEADS  # per-head dimension
rope = RotaryEmbedding(D_K, max_seq_len=BLOCK_SIZE)
print(f"RoPE dimension: d_k = {D_K}, max_seq_len = {BLOCK_SIZE}")

# ── 3.1: Plot rotation angles per frequency band ───────────────────────
# The rotation angles: theta_i = base^(-2i/d_k) for i = 0, 1, ..., d_k/2 - 1
# Different i = different rotation speed (frequency)
#   Small i (0, 1, ...) → LARGE theta  → cos changes fast → encodes LOCAL position
#   Large i (d/2-1, ...) → SMALL theta → cos changes slow → encodes LONG-RANGE position

n_freqs = D_K // 2  # number of frequency pairs
base = 10000.0
theta = base ** (-2.0 * torch.arange(n_freqs) / D_K)  # (n_freqs,)
positions = torch.arange(seq_len)

fig, axes = plt.subplots(2, 1, figsize=(10, 7))

# ── Top: cos(m*theta_i) for selected frequency bands ──────────────────
ax = axes[0]
band_indices = [0, 1, n_freqs // 4, n_freqs // 2 - 1]
band_indices = [i for i in band_indices if i < n_freqs]

for idx in band_indices:
    angle = positions * theta[idx]
    ax.plot(
        positions,
        torch.cos(angle),
        marker="o",
        markersize=3,
        label=rf"$i={idx}$, $\theta={theta[idx]:.4f}$",
    )
ax.set_xlabel("Position (m)")
ax.set_ylabel(r"$\cos(m \cdot \theta_i)$")
ax.set_title("RoPE Frequency Bands")
ax.legend()
ax.grid(True, alpha=0.3)

# ── Bottom: all frequency bands as a heatmap ──────────────────────────
ax = axes[1]
# cos(m * theta_i) for all i, all m — shape (T, n_freqs)
all_cos = torch.cos(positions[:, None] * theta[None, :])  # (T, n_freqs)
im = ax.imshow(
    all_cos.T,
    cmap="RdBu",
    aspect="auto",
    vmin=-1,
    vmax=1,
    extent=[0, seq_len - 1, n_freqs - 1, 0],
)
ax.set_xlabel("Position (m)")
ax.set_ylabel("Frequency Band Index (i)")
ax.set_title("RoPE Rotation Across All Frequency Bands")
fig.colorbar(im, ax=ax, label="cos(m·θᵢ)")

plt.tight_layout()
plt.show()

print(f"Frequency bands: {n_freqs}, d_k={D_K}")
print(
    f"  θ[0] = {theta[0]:.6f}  ← highest freq → encodes LOCAL position (changes every ~few tokens)"
)
print(f"  θ[{n_freqs // 4}] = {theta[n_freqs // 4]:.6f}")
print(
    f"  θ[{n_freqs - 1}] = {theta[-1]:.6f}  ← lowest freq → encodes LONG-RANGE position (changes every ~thousands of tokens)"
)

In [ ]:
# ── 3.2: RoPE rotation matrix over positions ──────────────────────────
# Visualise how RoPE makes the Q·K dot product depend on relative position.
#
# We create a unit vector at position m and another at position n, then
# compute their dot product AFTER applying RoPE. The result depends on
# (m-n) — NOT on m or n individually.

cos, sin = rope.cos, rope.sin  # precomputed tables: (max_seq_len, d_k)

# Create unit vectors
q_unit = torch.ones(1, D_K, dtype=torch.float32)
k_unit = torch.ones(1, D_K, dtype=torch.float32)

score_rope = torch.zeros(seq_len, seq_len)
score_identity = torch.ones(seq_len, seq_len)  # without RoPE: always 1.0

for m in range(seq_len):
    q_rot = apply_rotary_emb(q_unit, cos[m : m + 1], sin[m : m + 1])  # (1, d_k)
    for n in range(seq_len):
        k_rot = apply_rotary_emb(k_unit, cos[n : n + 1], sin[n : n + 1])  # (1, d_k)
        score_rope[m, n] = (q_rot @ k_rot.T).item()

# ── Plot ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: heatmap of RoPE-influenced dot product
ax = axes[0]
im = ax.imshow(
    score_rope,
    cmap="RdBu",
    aspect="auto",
    vmin=-score_rope.abs().max(),
    vmax=score_rope.abs().max(),
)
ax.set_title(
    r"RoPE Dot Product: $\langle \mathrm{RoPE}(q_m), \mathrm{RoPE}(k_n) \rangle$"
)
ax.set_xlabel("Key Position (n)")
ax.set_ylabel("Query Position (m)")
fig.colorbar(im, ax=ax)

# Right: contrast with identity (no RoPE)
ax = axes[1]
ax.imshow(score_identity, cmap="RdBu", aspect="auto", vmin=-1, vmax=1)
ax.set_title("Without RoPE: q·k = 1 (position-blind)")
ax.set_xlabel("Key Position (n)")
ax.set_ylabel("Query Position (m)")

plt.tight_layout()
plt.show()

print(f"With RoPE — diagonal (m=n) mean: {score_rope.diag().mean():.4f}")
mask = torch.eye(seq_len, dtype=torch.bool)
print(f"With RoPE — off-diagonal mean:   {score_rope[~mask].mean():.4f}")
print("Without RoPE: all entries = 1.0 (position-insensitive)")

In [ ]:
# ── 3.3: RoPE dot product as a function of distance ────────────────────
# Key property: RoPE(Q_m)·RoPE(K_n) depends ONLY on (m-n), not on
# absolute positions m and n. This is "translation invariance."

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: score_rope as a function of distance d = m - n
ax = axes[0]
distances = torch.arange(seq_len)
score_by_distance = torch.zeros(seq_len)
counts = torch.zeros(seq_len)

for m in range(seq_len):
    for n in range(m + 1):  # causal: n <= m
        d = m - n
        score_by_distance[d] += score_rope[m, n].item()
        counts[d] += 1

score_by_distance = score_by_distance / counts.clamp(min=1)

ax.plot(distances, score_by_distance, "o-", markersize=4, linewidth=1.5)
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Relative Distance d = m - n")
ax.set_ylabel("Average RoPE Dot Product")
ax.set_title("RoPE Dot Product vs Relative Distance\n(translation-invariant!)")
ax.grid(True, alpha=0.3)

# Right: verify translation invariance — overlay several rows
ax = axes[1]
for start_m in [0, seq_len // 4, seq_len // 2]:
    row = score_rope[start_m]  # fixed query position m
    # Plot row vs (m - n) shift
    shifted = torch.zeros(seq_len)
    weights = torch.zeros(seq_len)
    for n in range(seq_len):
        d = start_m - n
        if d >= 0:
            shifted[d] += row[n].item()
            weights[d] += 1
    shifted = shifted / weights.clamp(min=1)
    ax.plot(range(seq_len), shifted, "o-", markersize=3, label=f"query pos m={start_m}")

ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Distance (m - n)")
ax.set_ylabel("RoPE Dot Product")
ax.set_title(
    "Translation Invariance: Different Query Positions\nCollapse to Same Distance Curve"
)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight: RoPE(Q_m)·RoPE(K_n) = f(m-n)")
print(f"Self-attention (d=0): {score_by_distance[0]:.4f}")
print(f"Adjacent (d=1):       {score_by_distance[1]:.4f}" if seq_len > 1 else "")
print("The curve is identical regardless of which row (m) you pick!")

---
## 4. Interpreting What You See

### Attention Heads

| Pattern | Interpretation |
|---|---|
| **Strong diagonal** | Token mostly attends to itself — "what am I?" |
| **Previous-token peak** | Head learned to look 1 step back — syntactic bigram detection |
| **Broad uniform** | Head aggregates global context — "read all" |
| **Vertical stripe at position i** | Token i is universally attended — could be a sentence boundary |
| **Block structure** | Group of tokens attend to another group — phrase/chunk detection |

### RoPE

| Observation | Meaning |
|---|---|
| **Low-frequency bands change slowly** | Encode long-range position (100+ tokens) |
| **High-frequency bands change rapidly** | Encode local position (neighbouring tokens) |
| **Distance decay is smooth** | Model has a built-in locality prior without learned embeddings |
| **Translation invariance** | Same relative distance = same RoPE effect, regardless of absolute position |